# LeetCode 682 — Baseball Game bằng Python

**Link đề:** https://leetcode.com/problems/baseball-game/
**Độ khó:** Easy · **Chủ đề:** Array, Stack, Simulation

```python
class Solution:
    def calPoints(self, operations: list[str]) -> int:
        ...
```

## Mục tiêu học
1. Hiểu cấu trúc dữ liệu **stack** (ngăn xếp, LIFO) và cài bằng `list` Python: `append`, `pop`, `[-1]`, `[-2]`.
2. **Mô phỏng** (simulation) từng thao tác theo đúng luật đề.
3. Đổi chuỗi sang số với `int()` — kể cả số **âm** như `"-2"`.
4. Thấy vì sao cách "đánh dấu xóa rồi tìm lùi" chậm, và stack giải quyết gọn trong `O(n)`.
5. Biết thêm cách giữ **tổng chạy** (running total) và cú pháp `match/case` (Python 3.10+).

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Cell TODO không tự chạy.

## 1. Đề bài

Bạn ghi điểm cho một trận bóng chày với luật lạ. Bắt đầu với **bản ghi rỗng**. Mỗi phần tử `operations[i]` là một trong:

| Thao tác | Ý nghĩa |
|---|---|
| số nguyên `x` (dạng chuỗi) | ghi điểm mới `x` |
| `"+"` | ghi điểm mới = **tổng hai điểm gần nhất** |
| `"D"` | ghi điểm mới = **gấp đôi điểm gần nhất** |
| `"C"` | **hủy** điểm gần nhất (xóa khỏi bản ghi) |

Trả về **tổng** các điểm còn lại trong bản ghi.

**Ví dụ 1:** `["5","2","C","D","+"]` → `30`
```text
"5" -> [5]
"2" -> [5, 2]
"C" -> [5]          (hủy 2)
"D" -> [5, 10]      (2*5)
"+" -> [5, 10, 15]  (5+10)
Tổng = 30
```

**Ví dụ 2:** `["5","-2","4","C","D","9","+","+"]` → `27` (bản ghi cuối `[5, -2, -4, 9, 5, 14]`).

**Ví dụ 3:** `["1","C"]` → `0`.

**Constraints:**
- `1 <= len(operations) <= 1000`; số nguyên trong `[-3·10^4, 3·10^4]`.
- `"+"` luôn có ≥ 2 điểm trước đó; `"C"` và `"D"` luôn có ≥ 1 điểm trước đó.
- Kết quả và mọi phép tính trung gian vừa số nguyên 32-bit.

## 2. Nền tảng Python cần biết

### 2.1. Stack (ngăn xếp) — vào sau, ra trước (LIFO)
Hình dung **chồng đĩa**: đặt đĩa mới lên trên cùng, lấy ra cũng từ trên cùng.
Với `list` Python:
- `stack.append(x)` — đặt `x` lên đỉnh, `O(1)`.
- `stack.pop()` — lấy (và xóa) phần tử đỉnh, `O(1)`.
- `stack[-1]` — xem đỉnh (không xóa); `stack[-2]` — phần tử ngay dưới đỉnh.

In [1]:
stack = []
stack.append(5); print("append 5 ->", stack)
stack.append(2); print("append 2 ->", stack)
print("đỉnh stack[-1] =", stack[-1], "| dưới đỉnh stack[-2] =", stack[-2])
top = stack.pop(); print("pop ->", top, "| còn", stack)

append 5 -> [5]
append 2 -> [5, 2]
đỉnh stack[-1] = 2 | dưới đỉnh stack[-2] = 5
pop -> 2 | còn [5]


### 2.2. Tại sao bài này là stack?
Cả ba thao tác đặc biệt chỉ đụng tới **điểm gần nhất** (đỉnh) hoặc **hai điểm gần nhất**. "C" xóa điểm gần nhất → điểm **trước đó** trở thành "gần nhất" mới. Đó đúng là hành vi LIFO.

### 2.3. `int()` đổi chuỗi sang số (kể cả số âm)
`int("-2")` → `-2`. Đừng kiểm tra "là số" bằng `op.isdigit()` — `"-2".isdigit()` là `False`!

In [2]:
print(int("5"), int("-2"), int("30000"))
print('"5".isdigit() =', "5".isdigit(), '| "-2".isdigit() =', "-2".isdigit())

5 -2 30000
"5".isdigit() = True | "-2".isdigit() = False


### 2.4. `if / elif / else` theo thứ tự: kiểm tra ký hiệu trước, còn lại là số
Ba ký hiệu `"+"`, `"D"`, `"C"` được so trực tiếp; nhánh `else` xử lý mọi chuỗi số (dương hoặc âm).

In [3]:
def kind(op: str) -> str:
    if op == "+":
        return "cộng hai điểm gần nhất"
    elif op == "D":
        return "gấp đôi điểm gần nhất"
    elif op == "C":
        return "hủy điểm gần nhất"
    else:
        return f"số {int(op)}"

for op in ["5", "-2", "+", "D", "C"]:
    print(f"{op!r:>5}: {kind(op)}")

  '5': số 5
 '-2': số -2
  '+': cộng hai điểm gần nhất
  'D': gấp đôi điểm gần nhất
  'C': hủy điểm gần nhất


### 2.5. `sum(list)` — tổng các phần tử
`sum([])` = `0` → tự động xử lý trường hợp bản ghi rỗng (ví dụ 3).

In [4]:
print(sum([5, 10, 15]), sum([]))

30 0


## 3. Cách trực giác (kém hiệu quả) — giữ mọi điểm, đánh dấu "đã hủy"

Người mới đôi khi không xóa thật mà đánh dấu điểm bị hủy, rồi mỗi lần `"+"`/`"D"`/`"C"` phải **đi lùi** bỏ qua các điểm đã hủy để tìm điểm hợp lệ gần nhất.

Đúng, nhưng mỗi lần tìm lùi có thể tốn `O(n)` → tệ nhất `O(n²)`. Ngoài ra code dài và dễ sai.

In [5]:
def cal_points_marking(operations: list[str]) -> int:
    scores, alive = [], []          # alive[i] = điểm thứ i còn hiệu lực?

    def last_valid(k: int) -> list[int]:
        # Trả về chỉ số của k điểm hợp lệ gần nhất (đi lùi).
        res = []
        i = len(scores) - 1
        while i >= 0 and len(res) < k:
            if alive[i]:
                res.append(i)
            i -= 1
        return res

    for op in operations:
        if op == "+":
            a, b2 = last_valid(2)
            scores.append(scores[a] + scores[b2]); alive.append(True)
        elif op == "D":
            (a,) = last_valid(1)
            scores.append(2 * scores[a]); alive.append(True)
        elif op == "C":
            (a,) = last_valid(1)
            alive[a] = False
        else:
            scores.append(int(op)); alive.append(True)
    return sum(s for s, ok in zip(scores, alive) if ok)

print(cal_points_marking(["5", "2", "C", "D", "+"]))                    # 30
print(cal_points_marking(["5", "-2", "4", "C", "D", "9", "+", "+"]))    # 27
print(cal_points_marking(["1", "C"]))                                   # 0

30
27
0


## 4. Nút thắt và insight

**Nút thắt:** điểm bị hủy vẫn nằm trong list, làm việc tìm "điểm gần nhất" phải đi lùi qua chúng.

**Insight:** điểm bị hủy **luôn là điểm gần nhất** → nó nằm ở **cuối** list → xóa thật bằng `pop()` trong `O(1)`. Sau khi xóa, phần tử cuối mới chính là "điểm gần nhất" đúng nghĩa.

Vậy list luôn chỉ chứa điểm hợp lệ, và:
- `"+"` → `stack[-1] + stack[-2]`
- `"D"` → `2 * stack[-1]`
- `"C"` → `stack.pop()`
- số → `stack.append(int(op))`

Mỗi thao tác `O(1)` → tổng `O(n)`.

## 5. Lời giải chính — Stack (giảng từng đoạn)

**Đoạn 1:** `stack = []`.

**Đoạn 2:** duyệt `op` trong `operations`, chia 4 nhánh như insight.

**Đoạn 3:** trả `sum(stack)`.

Lưu ý thứ tự trong nhánh `"+"`: ta **đọc** `stack[-1]` và `stack[-2]` (không pop), rồi `append` tổng — hai điểm cũ vẫn ở lại bản ghi.

In [6]:
def cal_points_stack(operations: list[str]) -> int:
    stack = []                                  # đoạn 1
    for op in operations:                       # đoạn 2
        if op == "+":
            stack.append(stack[-1] + stack[-2])
        elif op == "D":
            stack.append(2 * stack[-1])
        elif op == "C":
            stack.pop()
        else:
            stack.append(int(op))
    return sum(stack)                           # đoạn 3

print(cal_points_stack(["5", "2", "C", "D", "+"]))
print(cal_points_stack(["5", "-2", "4", "C", "D", "9", "+", "+"]))
print(cal_points_stack(["1", "C"]))

30
27
0


## 6. TRACE từng bước

Ví dụ 2: `["5","-2","4","C","D","9","+","+"]`

| Bước | `op` | Hành động | `stack` sau |
|---:|---|---|---|
| 1 | `"5"` | push 5 | `[5]` |
| 2 | `"-2"` | push −2 | `[5, -2]` |
| 3 | `"4"` | push 4 | `[5, -2, 4]` |
| 4 | `"C"` | pop 4 | `[5, -2]` |
| 5 | `"D"` | push 2·(−2) = −4 | `[5, -2, -4]` |
| 6 | `"9"` | push 9 | `[5, -2, -4, 9]` |
| 7 | `"+"` | push −4 + 9 = 5 | `[5, -2, -4, 9, 5]` |
| 8 | `"+"` | push 9 + 5 = 14 | `[5, -2, -4, 9, 5, 14]` |

Tổng = 5 − 2 − 4 + 9 + 5 + 14 = **27**.

In [7]:
def trace_cal_points(operations: list[str]) -> int:
    stack = []
    for step, op in enumerate(operations, start=1):
        if op == "+":
            val = stack[-1] + stack[-2]
            action = f"push {stack[-2]} + {stack[-1]} = {val}"
            stack.append(val)
        elif op == "D":
            val = 2 * stack[-1]
            action = f"push 2*{stack[-1]} = {val}"
            stack.append(val)
        elif op == "C":
            action = f"pop {stack.pop()}"
        else:
            action = f"push {int(op)}"
            stack.append(int(op))
        print(f"bước {step}: op={op!r:<5} {action:<22} stack={stack}")
    print("tổng =", sum(stack))
    return sum(stack)

trace_cal_points(["5", "-2", "4", "C", "D", "9", "+", "+"])

bước 1: op='5'   push 5                 stack=[5]
bước 2: op='-2'  push -2                stack=[5, -2]
bước 3: op='4'   push 4                 stack=[5, -2, 4]
bước 4: op='C'   pop 4                  stack=[5, -2]
bước 5: op='D'   push 2*-2 = -4         stack=[5, -2, -4]
bước 6: op='9'   push 9                 stack=[5, -2, -4, 9]
bước 7: op='+'   push -4 + 9 = 5        stack=[5, -2, -4, 9, 5]
bước 8: op='+'   push 9 + 5 = 14        stack=[5, -2, -4, 9, 5, 14]
tổng = 27


27

In [8]:
trace_cal_points(["5", "2", "C", "D", "+"])

bước 1: op='5'   push 5                 stack=[5]
bước 2: op='2'   push 2                 stack=[5, 2]
bước 3: op='C'   pop 2                  stack=[5]
bước 4: op='D'   push 2*5 = 10          stack=[5, 10]
bước 5: op='+'   push 5 + 10 = 15       stack=[5, 10, 15]
tổng = 30


30

## 7. Code nộp LeetCode

### 7.1. Cách 1 — Stack (khuyên dùng)

In [9]:
class Solution:
    def calPoints(self, operations: list[str]) -> int:
        stack = []
        for op in operations:
            if op == "+":
                stack.append(stack[-1] + stack[-2])
            elif op == "D":
                stack.append(2 * stack[-1])
            elif op == "C":
                stack.pop()
            else:
                stack.append(int(op))
        return sum(stack)

print(Solution().calPoints(["5", "2", "C", "D", "+"]))
print(Solution().calPoints(["5", "-2", "4", "C", "D", "9", "+", "+"]))
print(Solution().calPoints(["1", "C"]))

30
27
0


### 7.2. Cách 2 — Stack + tổng chạy
Cập nhật `total` ngay khi push/pop: push `v` → `total += v`; pop → `total -= v`. Không cần `sum` cuối (vẫn `O(n)` tổng, nhưng hữu ích nếu phải trả lời tổng **sau mỗi thao tác**).

In [10]:
class SolutionRunning:
    def calPoints(self, operations: list[str]) -> int:
        stack, total = [], 0
        for op in operations:
            if op == "C":
                total -= stack.pop()
                continue
            if op == "+":
                v = stack[-1] + stack[-2]
            elif op == "D":
                v = 2 * stack[-1]
            else:
                v = int(op)
            stack.append(v)
            total += v
        return total

print(SolutionRunning().calPoints(["5", "-2", "4", "C", "D", "9", "+", "+"]))

27


### 7.3. Cách 3 — `match / case` (Python 3.10+)
`match` so khớp giá trị với từng `case`; `case _` là "mọi trường hợp còn lại" (như `else`).

In [11]:
class SolutionMatch:
    def calPoints(self, operations: list[str]) -> int:
        stack = []
        for op in operations:
            match op:
                case "+":
                    stack.append(stack[-1] + stack[-2])
                case "D":
                    stack.append(stack[-1] * 2)
                case "C":
                    stack.pop()
                case _:
                    stack.append(int(op))
        return sum(stack)

print(SolutionMatch().calPoints(["5", "2", "C", "D", "+"]))

30


## 8. Vì sao đúng?

**Bất biến:** sau khi xử lý `i` thao tác đầu, `stack` **bằng đúng** bản ghi điểm theo luật đề (theo thứ tự ghi, phần tử cuối là điểm gần nhất).

- Ban đầu: bản ghi rỗng, `stack = []`.
- Số `x`: bản ghi thêm `x` ở cuối ⇔ `append(x)`.
- `"+"`: hai điểm gần nhất là hai phần tử cuối `stack[-2], stack[-1]` → thêm tổng ⇔ `append(stack[-1] + stack[-2])`.
- `"D"`: điểm gần nhất là `stack[-1]` → thêm gấp đôi.
- `"C"`: xóa điểm gần nhất = phần tử cuối ⇔ `pop()`.

Đề đảm bảo luôn đủ phần tử cho `"+"`, `"D"`, `"C"` → không có `IndexError`. Cuối cùng `sum(stack)` = tổng bản ghi.

## 9. Độ phức tạp và so sánh

Gọi `n = len(operations)`.

| Cách | Time | Space | Ghi chú |
|---|---:|---:|---|
| Đánh dấu hủy + tìm lùi | `O(n²)` tệ nhất | `O(n)` | Dài, dễ sai |
| Stack | `O(n)` | `O(n)` | **Khuyên dùng** |
| Stack + tổng chạy | `O(n)` | `O(n)` | Có tổng sau mỗi bước |
| `match/case` | `O(n)` | `O(n)` | Cú pháp mới, cùng ý tưởng |

`int(op)` tốn thời gian theo độ dài chuỗi, nhưng chuỗi ≤ 6 ký tự nên coi là `O(1)`.

In [12]:
import time
ops_bad = ["1"] * 2000 + ["C"] * 1999 + ["D"] * 1000      # nhiều điểm bị hủy rồi D liên tục
for name, f in [("stack", Solution().calPoints), ("marking", cal_points_marking)]:
    t = time.perf_counter(); r = f(ops_bad)
    print(f"{name:>8}: tổng có {len(str(r))} chữ số  ({(time.perf_counter() - t) * 1000:.1f} ms)")

   stack: tổng có 302 chữ số  (0.9 ms)


 marking: tổng có 302 chữ số  (119.5 ms)


(Ví dụ trên cố tình vượt giới hạn 32-bit của đề chỉ để đo tốc độ — Python tự xử lý số lớn.) Cách đánh dấu chậm hơn hàng trăm lần vì mỗi `"D"` phải đi lùi qua gần 2000 điểm đã hủy.

## 10. Bộ test

1. Ba ví dụ đề.
2. Chỉ một số; số âm; `"C"` xóa hết.
3. `"+"` ngay sau `"C"` (dùng lại điểm cũ hơn).
4. `"D"` trên số âm.
5. Ngẫu nhiên hợp lệ: sinh dãy thao tác luôn đủ điều kiện, so stack với cách đánh dấu.

In [13]:
import random

cases = [
    (["5", "2", "C", "D", "+"], 30),
    (["5", "-2", "4", "C", "D", "9", "+", "+"], 27),
    (["1", "C"], 0),
    (["7"], 7),
    (["-30000"], -30000),
    (["3", "4", "C", "C"], 0),
    (["1", "2", "3", "C", "+"], 1 + 2 + 3),        # sau C còn [1,2] -> + = 3
    (["-3", "D", "D"], -3 - 6 - 12),
]
solvers = [Solution().calPoints, SolutionRunning().calPoints, SolutionMatch().calPoints,
           cal_points_stack, cal_points_marking]
for ops, expected in cases:
    for f in solvers:
        assert f(ops) == expected, (f, ops)
    print(f"{str(ops):<45} -> {expected} PASS")

def random_valid_ops(rng, n):
    ops, size = [], 0
    for _ in range(n):
        choices = ["num"]
        if size >= 1: choices += ["D", "C"]
        if size >= 2: choices += ["+"]
        c = rng.choice(choices)
        if c == "num":
            ops.append(str(rng.randint(-50, 50))); size += 1
        elif c == "C":
            ops.append("C"); size -= 1
        else:
            ops.append(c); size += 1
    return ops

rng = random.Random(682)
for _ in range(500):
    ops = random_valid_ops(rng, rng.randint(1, 20))
    exp = cal_points_marking(ops)
    assert all(f(ops) == exp for f in solvers)
print("500 dãy thao tác ngẫu nhiên hợp lệ: mọi cách khớp PASS")

['5', '2', 'C', 'D', '+']                     -> 30 PASS
['5', '-2', '4', 'C', 'D', '9', '+', '+']     -> 27 PASS
['1', 'C']                                    -> 0 PASS
['7']                                         -> 7 PASS
['-30000']                                    -> -30000 PASS
['3', '4', 'C', 'C']                          -> 0 PASS
['1', '2', '3', 'C', '+']                     -> 6 PASS
['-3', 'D', 'D']                              -> -21 PASS
500 dãy thao tác ngẫu nhiên hợp lệ: mọi cách khớp PASS


## 11. Lỗi người mới hay gặp

### Lỗi 1: `pop` hai điểm khi gặp `"+"`
`a = stack.pop(); b = stack.pop(); stack.append(a + b)` → làm **mất** hai điểm cũ khỏi bản ghi.

In [14]:
def wrong_plus_pop(operations):
    stack = []
    for op in operations:
        if op == "+":
            a = stack.pop(); b2 = stack.pop()          # SAI: xóa mất 2 điểm cũ
            stack.append(a + b2)
        elif op == "D":
            stack.append(2 * stack[-1])
        elif op == "C":
            stack.pop()
        else:
            stack.append(int(op))
    return sum(stack)

ops = ["5", "2", "C", "D", "+"]
print("sai:", wrong_plus_pop(ops), "| đúng:", Solution().calPoints(ops))

sai: 15 | đúng: 30


### Lỗi 2: Nhận diện số bằng `isdigit()`
`"-2".isdigit()` là `False` → số âm bị rơi vào nhánh khác (hoặc bỏ qua).

### Lỗi 3: Cộng thẳng `int(op)` vào tổng, quên "C" phải trừ đúng giá trị đã thêm
Nếu chỉ giữ biến `total` mà **không** có stack, khi gặp `"C"` ta không biết trừ bao nhiêu, cũng không biết "điểm gần nhất" mới là gì.

In [15]:
def wrong_isdigit(operations):
    stack = []
    for op in operations:
        if op.isdigit():                       # SAI với số âm
            stack.append(int(op))
        elif op == "+":
            stack.append(stack[-1] + stack[-2])
        elif op == "D":
            stack.append(2 * stack[-1])
        elif op == "C":
            stack.pop()
    return sum(stack)

ops = ["5", "-2", "4", "C", "D", "9", "+", "+"]
print("sai:", wrong_isdigit(ops), "| đúng:", Solution().calPoints(ops))

sai: 71 | đúng: 27


### Lỗi 4: Kiểm tra `else` trước các ký hiệu
Viết `int(op)` ở nhánh đầu tiên mà không kiểm tra → `int("+")` gây `ValueError`.

### Lỗi 5: Nhầm thứ tự `stack[-1]`, `stack[-2]` khi phép toán không giao hoán
Với `"+"` không sao (cộng giao hoán), nhưng ở biến thể có phép **trừ** (`"-"`) thì `stack[-2] - stack[-1]` khác `stack[-1] - stack[-2]`.

In [16]:
try:
    int("+")
except ValueError as e:
    print("ValueError:", e)

ValueError: invalid literal for int() with base 10: '+'


## 12. Mẹo nhận dạng pattern

- Thao tác chỉ đụng tới **phần tử gần nhất** / "hủy thao tác vừa rồi" (undo) → **stack**.
- Chuỗi lệnh dạng token (số, ký hiệu) cần xử lý tuần tự → **mô phỏng** với `if/elif` hoặc `match`.
- Bài cùng họ: 150 Evaluate Reverse Polish Notation, 20 Valid Parentheses, 1047 Remove All Adjacent Duplicates, 155 Min Stack, 1441 Build an Array With Stack Operations.

## 13. Biến thể

### 13.1. Evaluate Reverse Polish Notation (LeetCode 150)
Token là số → push; token là toán tử → **pop** hai số (lần này phải pop, vì toán hạng được "tiêu thụ"), tính, push kết quả. Chú ý thứ tự `a op b` với `b` là phần tử pop **trước**.

In [17]:
def eval_rpn(tokens: list[str]) -> int:
    stack = []
    for t in tokens:
        if t in {"+", "-", "*", "/"}:
            b2 = stack.pop()
            a = stack.pop()
            if t == "+": stack.append(a + b2)
            elif t == "-": stack.append(a - b2)
            elif t == "*": stack.append(a * b2)
            else: stack.append(int(a / b2))      # chia lấy phần nguyên về 0
        else:
            stack.append(int(t))
    return stack[0]

print(eval_rpn(["2", "1", "+", "3", "*"]))                                  # 9
print(eval_rpn(["4", "13", "5", "/", "+"]))                                 # 6
print(eval_rpn(["10", "6", "9", "3", "+", "-11", "*", "/", "*", "17", "+", "5", "+"]))  # 22

9
6
22


### 13.2. Thêm thao tác `"U"` (undo `"C"`)
Muốn khôi phục điểm vừa hủy → cần **stack thứ hai** lưu các điểm bị hủy.

In [18]:
def cal_points_with_undo(operations):
    stack, removed = [], []
    for op in operations:
        if op == "+": stack.append(stack[-1] + stack[-2])
        elif op == "D": stack.append(2 * stack[-1])
        elif op == "C": removed.append(stack.pop())
        elif op == "U": stack.append(removed.pop())
        else: stack.append(int(op))
    return sum(stack)

print(cal_points_with_undo(["5", "2", "C", "U", "+"]))   # [5,2] -> +7 -> 14

14


## 14. Bài tập (tự làm trước, đáp án ở cuối)

**Bài 1 — Trace tay:** `["3", "D", "+", "C", "-1", "+"]` — ghi `stack` sau mỗi bước và tổng.

**Bài 2 — Đoán output:**
```python
s = [1, 2, 3]
s.append(s[-1] * s[-2]); s.pop(0); print(s, s[-1], s[-3])
```

**Bài 3 — TODO:** trả về **danh sách tổng sau mỗi thao tác** (prefix totals).

In [19]:
def exercise_totals_after_each(operations: list[str]) -> list[int]:
    # TODO: dùng stack + total; sau mỗi op, append total vào kết quả
    pass

# assert exercise_totals_after_each(["5", "2", "C", "D", "+"]) == [5, 7, 5, 15, 30]

**Bài 4 — TODO:** thêm thao tác `"M"` = ghi điểm mới bằng **max** của toàn bản ghi hiện tại (đảm bảo bản ghi không rỗng). Yêu cầu `"M"` chạy `O(1)` — gợi ý: stack phụ lưu max.

In [20]:
def exercise_with_max(operations: list[str]) -> int:
    # TODO: giữ song song stack và max_stack (max_stack[-1] = max hiện tại)
    pass

# assert exercise_with_max(["1", "5", "M", "C", "C", "M"]) == 1 + 1
# assert exercise_with_max(["-1", "-5", "M", "+"]) == -1 - 5 - 1 - 6

**Bài 5 — TODO (LeetCode 1047):** xóa lặp lại các cặp ký tự **kề nhau giống nhau** trong chuỗi, trả chuỗi cuối.

In [21]:
def exercise_remove_adjacent(s: str) -> str:
    # TODO: stack ký tự; nếu đỉnh == ch thì pop, ngược lại push
    pass

# assert exercise_remove_adjacent("abbaca") == "ca"
# assert exercise_remove_adjacent("azxxzy") == "ay"

**Bài 6 — Mini challenge:** kiểm tra một dãy `operations` có **hợp lệ** không (mọi `"+"` có ≥ 2 điểm trước, mọi `"C"`/`"D"` có ≥ 1 điểm trước). Không cần tính điểm.

In [22]:
def exercise_is_valid_ops(operations: list[str]) -> bool:
    # TODO: chỉ cần theo dõi kích thước bản ghi
    pass

# assert exercise_is_valid_ops(["5", "2", "C", "D", "+"]) is True
# assert exercise_is_valid_ops(["5", "C", "D"]) is False
# assert exercise_is_valid_ops(["5", "+"]) is False

## 15. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Làm theo bảng mục 6 | Sau `"C"` stack là `[3, 6]` | `"+"` cuối = `6 + (-1)` |
| 2 | `append` trước, `pop(0)` sau | `pop(0)` xóa phần tử **đầu** | `[2, 3, 6]` |
| 3 | Dùng `SolutionRunning` | `res.append(total)` sau mỗi op | Kể cả op `"C"` |
| 4 | Max sau khi `C` phải "quay lại" max cũ | `max_stack.append(max(v, max_stack[-1]))` | pop cả hai stack khi `"C"` |
| 5 | Giống "C" hủy gần nhất | `if st and st[-1] == ch` | `"".join(st)` |
| 6 | Biến `size` | `"+"` cần `size >= 2` | `"C"` giảm size, số/`D`/`+` tăng size |

## 16. Đáp án tham khảo

**Bài 1:** `[3]` → `[3, 6]` → `[3, 6, 9]` → `[3, 6]` → `[3, 6, -1]` → `[3, 6, -1, 5]`; tổng `13`.

**Bài 2:** sau `append`: `[1, 2, 3, 6]`; sau `pop(0)`: `[2, 3, 6]` → in `[2, 3, 6] 6 2`.

In [23]:
trace_cal_points(["3", "D", "+", "C", "-1", "+"])
s = [1, 2, 3]
s.append(s[-1] * s[-2]); s.pop(0); print("Bài 2:", s, s[-1], s[-3])

bước 1: op='3'   push 3                 stack=[3]
bước 2: op='D'   push 2*3 = 6           stack=[3, 6]
bước 3: op='+'   push 3 + 6 = 9         stack=[3, 6, 9]
bước 4: op='C'   pop 9                  stack=[3, 6]
bước 5: op='-1'  push -1                stack=[3, 6, -1]
bước 6: op='+'   push 6 + -1 = 5        stack=[3, 6, -1, 5]
tổng = 13
Bài 2: [2, 3, 6] 6 2


In [24]:
def answer_totals_after_each(operations):
    stack, total, res = [], 0, []
    for op in operations:
        if op == "C":
            total -= stack.pop()
        else:
            if op == "+": v = stack[-1] + stack[-2]
            elif op == "D": v = 2 * stack[-1]
            else: v = int(op)
            stack.append(v); total += v
        res.append(total)
    return res


def answer_with_max(operations):
    stack, max_stack = [], []
    for op in operations:
        if op == "C":
            stack.pop(); max_stack.pop()
            continue
        if op == "+": v = stack[-1] + stack[-2]
        elif op == "D": v = 2 * stack[-1]
        elif op == "M": v = max_stack[-1]
        else: v = int(op)
        stack.append(v)
        max_stack.append(v if not max_stack else max(v, max_stack[-1]))
    return sum(stack)


def answer_remove_adjacent(s):
    st = []
    for ch in s:
        if st and st[-1] == ch:
            st.pop()
        else:
            st.append(ch)
    return "".join(st)


def answer_is_valid_ops(operations):
    size = 0
    for op in operations:
        if op == "+":
            if size < 2: return False
            size += 1
        elif op in ("C", "D"):
            if size < 1: return False
            size += 1 if op == "D" else -1
        else:
            size += 1
    return True


assert answer_totals_after_each(["5", "2", "C", "D", "+"]) == [5, 7, 5, 15, 30]
assert answer_with_max(["1", "5", "M", "C", "C", "M"]) == 1 + 1
assert answer_with_max(["-1", "-5", "M", "+"]) == -1 - 5 - 1 - 6
assert answer_remove_adjacent("abbaca") == "ca"
assert answer_remove_adjacent("azxxzy") == "ay"
assert answer_is_valid_ops(["5", "2", "C", "D", "+"]) is True
assert answer_is_valid_ops(["5", "C", "D"]) is False
assert answer_is_valid_ops(["5", "+"]) is False
for _ in range(300):
    ops = random_valid_ops(rng, rng.randint(1, 15))
    assert answer_is_valid_ops(ops)
    assert answer_totals_after_each(ops)[-1] == Solution().calPoints(ops)
print("Đáp án tham khảo: tất cả assert PASS")

Đáp án tham khảo: tất cả assert PASS


## 17. Template nhớ nhanh + checklist submit

```python
class Solution:
    def calPoints(self, operations: list[str]) -> int:
        stack = []
        for op in operations:
            if op == "+":
                stack.append(stack[-1] + stack[-2])   # đọc, KHÔNG pop
            elif op == "D":
                stack.append(2 * stack[-1])
            elif op == "C":
                stack.pop()
            else:
                stack.append(int(op))                 # int() xử lý cả số âm
        return sum(stack)
```

**Câu thần chú:** *"Gần nhất = đỉnh stack; hủy = pop; cộng/nhân đôi = đọc đỉnh rồi push."*

**Checklist:**
- [ ] Kiểm tra `"+"`, `"D"`, `"C"` trước, `else` mới `int(op)`.
- [ ] Không dùng `isdigit()` (sai với số âm).
- [ ] `"+"` chỉ **đọc** hai đỉnh, không pop.
- [ ] `sum([]) == 0` cho bản ghi rỗng.
- [ ] Time `O(n)`, Space `O(n)`.